# Day 4 — Prompt Evaluation Pipeline (Phase 2 Preview)

> **Note**: This notebook implements concepts from **Phase 2 (Evaluation)** 
> ahead of Phase 1 completion. I encountered prompt evaluation in course 
> material directly after prompt engineering, so I built a working 
> pipeline before formally entering Phase 2. The pipeline runs end-to-end 
> but has known scope cuts (see "What I deferred" below).

## What I built

A complete prompt evaluation pipeline with 5 stages:

1. **Draft a prompt** — initial template, leaves a `{task}` placeholder
2. **Generate eval dataset** — using Claude (Haiku) to produce 3 AWS-related 
   tasks (Python / JSON / Regex), saved as `dataset.json`
3. **Run prompt against dataset** — `run_prompt` → `run_test_case` → `run_eval` 
   pipeline; collects outputs for every test case
4. **Grade with LLM-as-judge** — `grade_by_model` uses Claude as evaluator, 
   returns structured JSON `{strengths, weaknesses, reasoning, score}`
5. **Aggregate** — average score across all test cases (got 7.17/10 baseline)

## What I learned today

- **Prompt evaluation is "ML for prompts"**: just as ML models need test 
  sets and metrics, prompts need eval datasets and scoring. Intuition 
  ("looks better to me") doesn't scale — you need data.
- **Prefilling response with stop_sequence forces structured output**: 
  by sending `{"role": "assistant", "content": "```json"}` then stopping 
  at `"```"`, you guarantee Claude returns JSON between those markers — 
  no markdown wrapper, no preamble, parseable on first try.
- **Three grader types exist**, each with tradeoffs:
  - **Model grader** (LLM-as-judge): scalable, nuanced, but expensive 
    and the judge can be biased toward its own writing style
  - **Code grader**: deterministic, fast, free — but only works for 
    objectively-checkable properties (syntax validity, JSON schema match)
  - **Human grader**: most accurate, doesn't scale
- **A grader is itself a prompt** — `grade_by_model` is just another 
  Claude call. This means graders need eval too (Phase 2 will revisit 
  meta-evaluation).
- **Stop sequences differ from max_tokens**: max_tokens caps length 
  blindly; stop_sequence stops at semantic boundaries (closing code fence, 
  end of structure). Both apply in this pipeline — max_tokens as safety, 
  stop_sequence as structure.

## Design choices I made + reasoning

- **Dataset from Claude (Haiku), not hand-written**: faster iteration, 
  but introduces a risk — the model that generates eval data may also be 
  the model under test, creating implicit bias. Hand-written datasets 
  are gold standard; generated ones are bootstrap.
- **3 test cases, not 30+**: enough to verify the pipeline works, not 
  enough for statistical claims. Phase 2 capstone will scale this.
- **Generic "expert code reviewer" grader prompt**: did not yet 
  decompose into the 3 explicit criteria (Format / Syntax / Task 
  Following) that the design intended. Current grader is holistic; 
  future version should be criteria-specific.
- **Average score, not per-task breakdown**: aggregate masks high-variance 
  cases. The regex test scored 6, the S3 validator scored 8.5 — averaging 
  hides this signal. Should surface min/max/std as well.

## What I deferred (honest scope cuts)

- ❌ **Code grader** — designed in Step 4 markdown, not implemented. 
  Would check JSON validity, Python AST parsability, regex compileability.
- ❌ **Human grader** — out of scope for a sandbox notebook.
- ❌ **Step 5: prompt iteration** — only ran baseline. No A/B comparison 
  yet. This is the actual learning loop ("did my prompt change improve 
  the score?") and is the highest-value follow-up.
- ❌ **3-criteria scoring breakdown** — current grader gives one holistic 
  score, not separate Format/Syntax/Task-Following scores.

---

<Interview Q&A — Prompt Evaluation>

### Q1: What's the difference between prompt engineering and prompt evaluation?

**Prompt engineering** is the craft of *writing* prompts — choosing 
techniques (zero-shot, few-shot, CoT, XML tags) to elicit desired 
behavior from an LLM. It's creative and exploratory.

**Prompt evaluation** is the discipline of *measuring* prompt quality — 
building test datasets, running prompts across them, scoring outputs, 
and detecting regressions. It's quantitative and reproducible.

Without evaluation, prompt engineering is "I think this version is 
better." With evaluation, it's "version B scores 8.2/10 vs version A's 
7.1/10 on 50 test cases (p < 0.05)." Evaluation is what separates 
hobbyist prompts from production prompts.

### Q2: What are the three grader types? When does each apply?

| Grader | How it works | Use when |
|---|---|---|
| **Model grader** (LLM-as-judge) | Use another LLM to score the output, often returning structured JSON with reasoning | Output quality is nuanced and not mechanically checkable (writing quality, helpfulness, accuracy on open-ended tasks). Most flexible, scales well, but expensive and can be biased. |
| **Code grader** | Programmatic checks: regex match, schema validation, AST parsability, key field presence | Output has objective correctness properties (valid JSON, syntactically correct Python, contains required fields). Fast, free, deterministic. Limited to checkable properties. |
| **Human grader** | A person manually scores each output | Ground-truth establishment, calibrating other graders, high-stakes domains (medical, legal). Doesn't scale beyond ~100 examples per round. |

**Production pattern**: combine all three. Code grader catches the 
obvious failures (filters), model grader scores the rest (assessment), 
human grader validates a sample (calibration).

### Q3: Why use prefilling + stop_sequence for structured output?

The naive way to get JSON from Claude:

In [2]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()

import json

In [11]:
### Step 1: Draft a prompt 
# prompt = f"""
# Please provide a solution to the following task:

# {task}
# """

In [4]:
### Step 2: Generate eval dataset

#### 3 helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text


#### the 4th helper function - dataset generation function
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to 
evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related 
tasks. Generate an array of JSON objects, each representing task that requires Python, 
JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```


* Focus on tasks that can be solved by writing a single Python function, a single 
JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages=messages, text=prompt)
    add_assistant_message(messages=messages, text="```json")
    text = chat(messages, system=None, temperature=1.0, stop_sequences=["```"])
    return json.loads(text)


In [ ]:
#### generate the dataset 
model = "claude-haiku-4-5-20251001"
dataset = generate_dataset()
dataset

[{'task': 'Write a Python function that takes an AWS S3 bucket name and returns True if it follows AWS naming conventions (lowercase, 3-63 characters, starts with letter or number, no consecutive hyphens), False otherwise.'},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to all objects in an S3 bucket named 'my-data-bucket'."},
 {'task': 'Write a regex pattern that matches valid AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters).'}]

In [19]:
#### save the dataset
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [33]:
### Step 3: Feed through Claude and run eval 

#### core function 1: run_prompt()
def run_prompt(test_case):
    # Merges the initial prompt and test case input, then returns the result
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages=messages, text=prompt)
    output = chat(messages)
    return output
    

#### core function 2: run_test_case()
def run_test_case(test_case):
    # run run_prompt for the test case and then grade the result
    output = run_prompt(test_case)

    # ToDo - Grade
    score = 10.0

    # assemble and return result
    result = {
        "test_case": test_case,
        "output": output,
        "score": score
    }
    return result


#### core function 3: run_eval()
def run_eval(dataset):
    # load dataset and run run_test_case for each test_case, return a result list
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [34]:
#### load dataset and run run_eval
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [35]:
#### Examine the result
print(json.dumps(results, indent=2))

[
  {
    "test_case": {
      "task": "Write a Python function that takes an AWS S3 bucket name and returns True if it follows AWS naming conventions (lowercase, 3-63 characters, starts with letter or number, no consecutive hyphens), False otherwise."
    },
    "output": "# AWS S3 Bucket Naming Convention Validator\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates if a bucket name follows AWS S3 naming conventions.\n    \n    AWS S3 bucket naming rules:\n    - Must be between 3 and 63 characters long\n    - Must contain only lowercase letters, numbers, and hyphens\n    - Must start with a lowercase letter or number\n    - Cannot have consecutive hyphens\n    - Cannot end with a hyphen\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name (str): The S3 bucket name to validate\n        \n    Returns:\n        bool: True if valid, False otherwise\n    \"\"\"\n    \n    # Check if b

In [41]:
### Step 4: Feed through Graders
#### step 4a - model grader
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

#### Integrating Grading into Your Workflow
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }


#### from statistics import mean
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [42]:
#### load dataset and run run_eval
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.166666666666667


In [43]:
#### Examine the result
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Naming Convention Validator\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates if a bucket name follows AWS S3 naming conventions.\n    \n    AWS S3 bucket naming rules:\n    - Must be between 3 and 63 characters long\n    - Must start with a lowercase letter or number\n    - Can contain only lowercase letters, numbers, and hyphens\n    - Cannot have consecutive hyphens\n    - Cannot end with a hyphen\n    - Cannot be formatted as an IP address (x.x.x.x)\n    \n    Args:\n        bucket_name: The S3 bucket name to validate\n        \n    Returns:\n        True if the bucket name follows AWS naming conventions, False otherwise\n    \"\"\"\n    \n    # Check if bucket_name is a string\n    if not isinstance(bucket_name, str):\n        return False\n    \n    # Check length (3-63 characters)\n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return False\n    \n    # Check if starts